In [2]:
"""
Action Recognition Inference
==============================
Set VIDEO_PATH below and run:
    python predict_action.py

Requirements:
    pip install ultralytics tensorflow opencv-python numpy scikit-learn joblib
"""

import cv2
import numpy as np
import torch
from ultralytics import YOLO
from tensorflow.keras.models import load_model
from pathlib import Path
import joblib

# ─────────────────────────────────────────
# CONFIG — change these
# ─────────────────────────────────────────
VIDEO_PATH        = "data/lung2.mp4"   # <-- change this
LSTM_MODEL_PATH   = "pose_lstm_model.h5"
SCALER_PATH       = "scaler.pkl"
ACTION_NAMES_PATH = "action_names_filtered.npy"
YOLO_MODEL        = "yolov8n-pose.pt"
SEQ_LEN           = 60
SUBSAMPLE         = 5
NUM_SAMPLES       = 5     # evenly spaced clips to sample; increase for longer vids
KP_CONF_THRESH    = 0.5   # skip frames with low keypoint confidence (must match training)
# ─────────────────────────────────────────


def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = (np.arctan2(c[1]-b[1], c[0]-b[0])
             - np.arctan2(a[1]-b[1], a[0]-b[0]))
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0: angle = 360 - angle
    return angle


def get_body_orientation(kp):
    mid_shoulder = (kp[5] + kp[6]) / 2
    mid_hip      = (kp[11] + kp[12]) / 2
    spine_vec    = mid_shoulder - mid_hip
    return np.degrees(np.arctan2(abs(spine_vec[0]), abs(spine_vec[1]) + 1e-6))


def get_torso_lean(kp):
    mid_shoulder = (kp[5] + kp[6]) / 2
    mid_hip      = (kp[11] + kp[12]) / 2
    spine_vec    = mid_shoulder - mid_hip
    return spine_vec[0] / (abs(spine_vec[1]) + 1e-6)


def get_arm_angles(kp):
    mid_hip      = (kp[11] + kp[12]) / 2
    mid_shoulder = (kp[5]  + kp[6])  / 2
    torso_vec    = mid_hip - mid_shoulder
    l_arm_vec    = kp[7] - kp[5]
    r_arm_vec    = kp[8] - kp[6]
    def vec_angle(v1, v2):
        cos = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-6)
        return np.degrees(np.arccos(np.clip(cos, -1, 1)))
    return vec_angle(torso_vec, l_arm_vec), vec_angle(torso_vec, r_arm_vec)


def get_hip_knee_alignment(kp):
    hip_width  = np.linalg.norm(kp[11] - kp[12]) + 1e-6
    knee_width = np.linalg.norm(kp[13] - kp[14])
    return knee_width / hip_width


def get_wrist_hip_height(kp, mid_hip):
    return (mid_hip[1] - kp[9][1]), (mid_hip[1] - kp[10][1])


def extract_features_from_frame(kp, prev_kp=None):
    mid_hip = (kp[11] + kp[12]) / 2
    norm_kp = kp - mid_hip

    l_elbow = calculate_angle(kp[5], kp[7], kp[9])
    r_elbow = calculate_angle(kp[6], kp[8], kp[10])
    l_knee  = calculate_angle(kp[11], kp[13], kp[15])
    r_knee  = calculate_angle(kp[12], kp[14], kp[16])

    orientation    = get_body_orientation(kp)
    torso_lean     = get_torso_lean(kp)
    l_abd, r_abd   = get_arm_angles(kp)
    knee_hip_ratio = get_hip_knee_alignment(kp)
    l_wh, r_wh     = get_wrist_hip_height(kp, mid_hip)
    l_ext          = np.linalg.norm(kp[9]  - kp[5])
    r_ext          = np.linalg.norm(kp[10] - kp[6])

    if prev_kp is not None:
        l_wrist_vel = np.linalg.norm(kp[9]  - prev_kp[9])
        r_wrist_vel = np.linalg.norm(kp[10] - prev_kp[10])
        l_ankle_vel = np.linalg.norm(kp[15] - prev_kp[15])
        r_ankle_vel = np.linalg.norm(kp[16] - prev_kp[16])
    else:
        l_wrist_vel = r_wrist_vel = l_ankle_vel = r_ankle_vel = 0.0

    return np.concatenate([
        norm_kp.flatten(),
        [l_elbow, r_elbow, l_knee, r_knee],
        [orientation, torso_lean],
        [l_abd, r_abd],
        [knee_hip_ratio],
        [l_wh, r_wh],
        [l_ext, r_ext],
        [l_wrist_vel, r_wrist_vel, l_ankle_vel, r_ankle_vel],
    ])  # 51 features


def extract_all_features(video_path, yolo_model):
    """Run YOLO on every frame, filter low-confidence, return subsampled features."""
    device = 0 if torch.cuda.is_available() else "cpu"
    cap    = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    raw_features  = []
    prev_kp       = None
    frame_count   = 0
    skipped       = 0

    print("Extracting pose features", end="", flush=True)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1
        if frame_count % 50 == 0:
            print(".", end="", flush=True)

        results  = yolo_model(frame, verbose=False, device=device)
        kp_data  = results[0].keypoints

        if kp_data is None or len(kp_data.xy) == 0:
            skipped += 1
            continue

        kp   = kp_data.xy[0].cpu().numpy()
        conf = kp_data.conf[0].cpu().numpy()

        if conf.mean() < KP_CONF_THRESH:
            skipped += 1
            continue

        feat = extract_features_from_frame(kp, prev_kp)
        raw_features.append(feat)
        prev_kp = kp

    cap.release()
    print(f"\nFrames: {frame_count} | Pose frames kept: {len(raw_features)} | Skipped: {skipped}")
    return raw_features[::SUBSAMPLE]


def build_samples(filtered_frames):
    """Pull NUM_SAMPLES evenly spaced 60-frame windows from the video."""
    total = len(filtered_frames)

    if total < SEQ_LEN:
        indices  = np.linspace(0, total - 1, SEQ_LEN).astype(int)
        sequence = np.array([filtered_frames[i] for i in indices])
        print(f"Short video: 1 sample (interpolated from {total} frames)")
        return sequence[np.newaxis, ...]

    max_start    = total - SEQ_LEN
    start_points = np.linspace(0, max_start, NUM_SAMPLES).astype(int)
    samples      = [np.array(filtered_frames[s:s + SEQ_LEN]) for s in start_points]
    print(f"Pulled {len(samples)} samples from {total} subsampled frames")
    return np.array(samples)   # (NUM_SAMPLES, SEQ_LEN, 51)


def apply_scaler(X, scaler):
    """Apply saved scaler to (N, SEQ_LEN, 51) array."""
    n, t, f = X.shape
    return scaler.transform(X.reshape(-1, f)).reshape(n, t, f)


def predict():
    print(f"\nVideo      : {VIDEO_PATH}")
    print(f"Model      : {LSTM_MODEL_PATH}")
    print(f"Scaler     : {SCALER_PATH}\n")

    # Load everything
    print("Loading models...")
    yolo         = YOLO(YOLO_MODEL)
    lstm         = load_model(LSTM_MODEL_PATH)
    scaler       = joblib.load(SCALER_PATH)
    action_names = np.load(ACTION_NAMES_PATH)

    # Extract
    filtered_frames = extract_all_features(str(Path(VIDEO_PATH).resolve()), yolo)

    if len(filtered_frames) < 10:
        print("Not enough usable frames to classify.")
        return

    # Build samples → scale → predict
    X          = build_samples(filtered_frames)
    X_scaled   = apply_scaler(X, scaler)
    all_probs  = lstm.predict(X_scaled, verbose=0)

    # Per-sample results
    print("\n--- Per-sample predictions ---")
    for i, probs in enumerate(all_probs):
        pred_idx   = int(np.argmax(probs))
        confidence = float(probs[pred_idx]) * 100
        print(f"  Sample {i+1:>2}: {action_names[pred_idx]:<20} ({confidence:.1f}%)")

    # Average across samples → final answer
    avg_probs  = np.mean(all_probs, axis=0)
    pred_idx   = int(np.argmax(avg_probs))
    confidence = float(avg_probs[pred_idx]) * 100

    print("\n" + "=" * 45)
    print(f"  Final Predicted Action : {action_names[pred_idx]}")
    print(f"  Confidence (avg)       : {confidence:.1f}%")
    print("=" * 45)

    print("\nAll class probabilities:")
    for name, prob in sorted(zip(action_names, avg_probs), key=lambda x: x[1], reverse=True):
        bar = "#" * int(prob * 30)
        print(f"  {name:<20} {prob*100:5.1f}%  {bar}")


if __name__ == "__main__":
    predict()


Video      : data/lung2.mp4
Model      : pose_lstm_model.h5
Scaler     : scaler.pkl

Loading models...
Extracting pose features

/Users/sushanth/Library/jupyterlab-desktop/envs/tenso/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


.......................
Frames: 1193 | Pose frames kept: 766 | Skipped: 427
Pulled 5 samples from 154 subsampled frames

--- Per-sample predictions ---
  Sample  1: BodyWeightSquats     (99.7%)
  Sample  2: BodyWeightSquats     (98.6%)
  Sample  3: JumpingJack          (100.0%)
  Sample  4: Lunges               (100.0%)
  Sample  5: Lunges               (100.0%)

  Final Predicted Action : Lunges
  Confidence (avg)       : 40.3%

All class probabilities:
  Lunges                40.3%  ############
  BodyWeightSquats      39.7%  ###########
  JumpingJack           20.0%  ######
  BenchPress             0.0%  
  PushUps                0.0%  
  PullUps                0.0%  


In [1]:
"""
Real-time Action Recognition — Video File or Live Webcam
==========================================================
Set MODE below:
  - "video"  : processes a video file, shows frame-by-frame with overlay
  - "webcam" : uses your webcam (device 0) for live inference

Run:
    python visualize_prediction.py

Requirements:
    pip install ultralytics tensorflow opencv-python numpy scikit-learn joblib
"""

import cv2
import numpy as np
import torch
import joblib
import collections
from ultralytics import YOLO
from tensorflow.keras.models import load_model
from pathlib import Path

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
MODE              = "webcam"                  # "video" or "webcam"
VIDEO_PATH        = "data/lung1.mp4" # only used when MODE = "video"
LSTM_MODEL_PATH   = "pose_lstm_model.h5"
SCALER_PATH       = "scaler.pkl"
ACTION_NAMES_PATH = "action_names_filtered.npy"
YOLO_MODEL        = "yolov8n-pose.pt"
SEQ_LEN           = 60
SUBSAMPLE         = 5                        # run prediction every Nth frame
KP_CONF_THRESH    = 0.5
SMOOTHING_WINDOW  = 10                       # average predictions over last N results
# ─────────────────────────────────────────

# Skeleton connections (COCO 17-keypoint format)
SKELETON = [
    (0, 1), (0, 2), (1, 3), (2, 4),          # head
    (5, 6),                                    # shoulders
    (5, 7), (7, 9),                            # left arm
    (6, 8), (8, 10),                           # right arm
    (5, 11), (6, 12),                          # torso sides
    (11, 12),                                  # hips
    (11, 13), (13, 15),                        # left leg
    (12, 14), (14, 16),                        # right leg
]

# Colors (BGR)
COL_SKELETON   = (0, 255, 180)
COL_KEYPOINT   = (255, 255, 255)
COL_LOW_CONF   = (80, 80, 80)
COL_BAR_BG     = (40, 40, 40)
COL_BAR_FILL   = (0, 200, 120)
COL_BAR_TOP    = (0, 255, 180)
COL_TEXT       = (255, 255, 255)
COL_LABEL      = (180, 180, 180)
COL_PANEL_BG   = (20, 20, 20)
COL_SKIPPED    = (60, 60, 200)
PANEL_W        = 300


def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = (np.arctan2(c[1]-b[1], c[0]-b[0])
             - np.arctan2(a[1]-b[1], a[0]-b[0]))
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0: angle = 360 - angle
    return angle

def get_body_orientation(kp):
    mid_shoulder = (kp[5] + kp[6]) / 2
    mid_hip      = (kp[11] + kp[12]) / 2
    spine_vec    = mid_shoulder - mid_hip
    return np.degrees(np.arctan2(abs(spine_vec[0]), abs(spine_vec[1]) + 1e-6))

def get_torso_lean(kp):
    mid_shoulder = (kp[5] + kp[6]) / 2
    mid_hip      = (kp[11] + kp[12]) / 2
    spine_vec    = mid_shoulder - mid_hip
    return spine_vec[0] / (abs(spine_vec[1]) + 1e-6)

def get_arm_angles(kp):
    mid_hip      = (kp[11] + kp[12]) / 2
    mid_shoulder = (kp[5]  + kp[6])  / 2
    torso_vec    = mid_hip - mid_shoulder
    l_arm_vec    = kp[7] - kp[5]
    r_arm_vec    = kp[8] - kp[6]
    def vec_angle(v1, v2):
        cos = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-6)
        return np.degrees(np.arccos(np.clip(cos, -1, 1)))
    return vec_angle(torso_vec, l_arm_vec), vec_angle(torso_vec, r_arm_vec)

def get_hip_knee_alignment(kp):
    hip_width  = np.linalg.norm(kp[11] - kp[12]) + 1e-6
    knee_width = np.linalg.norm(kp[13] - kp[14])
    return knee_width / hip_width

def get_wrist_hip_height(kp, mid_hip):
    return (mid_hip[1] - kp[9][1]), (mid_hip[1] - kp[10][1])

def extract_features_from_frame(kp, prev_kp=None):
    mid_hip = (kp[11] + kp[12]) / 2
    norm_kp = kp - mid_hip
    l_elbow = calculate_angle(kp[5], kp[7], kp[9])
    r_elbow = calculate_angle(kp[6], kp[8], kp[10])
    l_knee  = calculate_angle(kp[11], kp[13], kp[15])
    r_knee  = calculate_angle(kp[12], kp[14], kp[16])
    orientation    = get_body_orientation(kp)
    torso_lean     = get_torso_lean(kp)
    l_abd, r_abd   = get_arm_angles(kp)
    knee_hip_ratio = get_hip_knee_alignment(kp)
    l_wh, r_wh     = get_wrist_hip_height(kp, mid_hip)
    l_ext          = np.linalg.norm(kp[9]  - kp[5])
    r_ext          = np.linalg.norm(kp[10] - kp[6])
    if prev_kp is not None:
        l_wrist_vel = np.linalg.norm(kp[9]  - prev_kp[9])
        r_wrist_vel = np.linalg.norm(kp[10] - prev_kp[10])
        l_ankle_vel = np.linalg.norm(kp[15] - prev_kp[15])
        r_ankle_vel = np.linalg.norm(kp[16] - prev_kp[16])
    else:
        l_wrist_vel = r_wrist_vel = l_ankle_vel = r_ankle_vel = 0.0
    return np.concatenate([
        norm_kp.flatten(),
        [l_elbow, r_elbow, l_knee, r_knee],
        [orientation, torso_lean],
        [l_abd, r_abd],
        [knee_hip_ratio],
        [l_wh, r_wh],
        [l_ext, r_ext],
        [l_wrist_vel, r_wrist_vel, l_ankle_vel, r_ankle_vel],
    ])


def draw_skeleton(frame, kp, conf):
    """Draw skeleton and keypoints on frame."""
    # Connections
    for i, j in SKELETON:
        if conf[i] > KP_CONF_THRESH and conf[j] > KP_CONF_THRESH:
            pt1 = (int(kp[i][0]), int(kp[i][1]))
            pt2 = (int(kp[j][0]), int(kp[j][1]))
            cv2.line(frame, pt1, pt2, COL_SKELETON, 2, cv2.LINE_AA)
    # Keypoints
    for i in range(len(kp)):
        if conf[i] > KP_CONF_THRESH:
            pt  = (int(kp[i][0]), int(kp[i][1]))
            col = COL_KEYPOINT if conf[i] > 0.7 else COL_LOW_CONF
            cv2.circle(frame, pt, 4, col, -1, cv2.LINE_AA)
            cv2.circle(frame, pt, 5, COL_SKELETON, 1, cv2.LINE_AA)


def draw_panel(frame_h, action_names, avg_probs, top_action, confidence, frame_count, buffer_size):
    """Build the side prediction panel."""
    panel = np.full((frame_h, PANEL_W, 3), COL_PANEL_BG, dtype=np.uint8)

    # Title
    cv2.putText(panel, "ACTION RECOGNITION", (12, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, COL_TEXT, 1, cv2.LINE_AA)
    cv2.line(panel, (12, 38), (PANEL_W - 12, 38), (60, 60, 60), 1)

    # Top prediction
    cv2.putText(panel, top_action, (12, 72),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, COL_BAR_TOP, 2, cv2.LINE_AA)
    cv2.putText(panel, f"{confidence:.1f}%", (12, 96),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, COL_LABEL, 1, cv2.LINE_AA)

    cv2.line(panel, (12, 108), (PANEL_W - 12, 108), (60, 60, 60), 1)

    # Probability bars — sorted by probability
    bar_y     = 120
    bar_h     = 28
    bar_gap   = 10
    max_bar_w = PANEL_W - 24

    sorted_pairs = sorted(zip(action_names, avg_probs), key=lambda x: x[1], reverse=True)

    for name, prob in sorted_pairs:
        fill_w = int(prob * max_bar_w)
        is_top = (name == top_action)

        # Background bar
        cv2.rectangle(panel, (12, bar_y), (12 + max_bar_w, bar_y + bar_h),
                      COL_BAR_BG, -1)
        # Fill
        bar_col = COL_BAR_TOP if is_top else COL_BAR_FILL
        if fill_w > 0:
            cv2.rectangle(panel, (12, bar_y), (12 + fill_w, bar_y + bar_h),
                          bar_col, -1)
        # Label
        label_col = COL_TEXT if is_top else COL_LABEL
        cv2.putText(panel, name, (16, bar_y + 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, label_col, 1, cv2.LINE_AA)
        # Percentage
        pct_str = f"{prob*100:.0f}%"
        cv2.putText(panel, pct_str, (PANEL_W - 42, bar_y + 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, label_col, 1, cv2.LINE_AA)

        bar_y += bar_h + bar_gap

    # Buffer fill indicator
    buf_y  = frame_h - 70
    buf_pct = min(buffer_size / SEQ_LEN, 1.0)
    cv2.putText(panel, "Buffer", (12, buf_y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, COL_LABEL, 1, cv2.LINE_AA)
    cv2.rectangle(panel, (12, buf_y + 8), (PANEL_W - 12, buf_y + 20), COL_BAR_BG, -1)
    cv2.rectangle(panel, (12, buf_y + 8),
                  (12 + int(buf_pct * (PANEL_W - 24)), buf_y + 20), (80, 130, 200), -1)
    cv2.putText(panel, f"{buffer_size}/{SEQ_LEN} frames", (12, buf_y + 36),
                cv2.FONT_HERSHEY_SIMPLEX, 0.38, COL_LABEL, 1, cv2.LINE_AA)

    # Frame counter
    cv2.putText(panel, f"Frame: {frame_count}", (12, frame_h - 16),
                cv2.FONT_HERSHEY_SIMPLEX, 0.38, (80, 80, 80), 1, cv2.LINE_AA)

    return panel


def run():
    device = 0 if torch.cuda.is_available() else "cpu"

    print("Loading models...")
    yolo         = YOLO(YOLO_MODEL)
    lstm         = load_model(LSTM_MODEL_PATH)
    scaler       = joblib.load(SCALER_PATH)
    action_names = list(np.load(ACTION_NAMES_PATH))
    num_classes  = len(action_names)

    source = 0 if MODE == "webcam" else VIDEO_PATH
    cap    = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open source: {source}")

    print(f"Running in {MODE.upper()} mode — press Q to quit")

    # Rolling feature buffer and prediction smoother
    feature_buffer  = collections.deque(maxlen=SEQ_LEN)
    prob_smoother   = collections.deque(maxlen=SMOOTHING_WINDOW)
    avg_probs       = np.ones(num_classes) / num_classes
    top_action      = "Waiting..."
    confidence      = 0.0
    frame_count     = 0
    prev_kp         = None
    subsample_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count     += 1
        subsample_count += 1
        display          = frame.copy()

        results  = yolo(frame, verbose=False, device=device)
        kp_data  = results[0].keypoints

        pose_detected = False
        if kp_data is not None and len(kp_data.xy) > 0:
            kp   = kp_data.xy[0].cpu().numpy()
            conf = kp_data.conf[0].cpu().numpy()

            # Draw skeleton regardless of confidence
            draw_skeleton(display, kp, conf)

            # Only add to buffer if frame quality is good
            if conf.mean() >= KP_CONF_THRESH and subsample_count >= SUBSAMPLE:
                feat = extract_features_from_frame(kp, prev_kp)
                feature_buffer.append(feat)
                prev_kp         = kp
                subsample_count = 0
                pose_detected   = True

        # Run prediction whenever buffer is full
        if len(feature_buffer) == SEQ_LEN:
            seq    = np.array(feature_buffer)                        # (60, 51)
            n, f   = seq.shape
            scaled = scaler.transform(seq).reshape(1, n, f)
            probs  = lstm.predict(scaled, verbose=0)[0]
            prob_smoother.append(probs)
            avg_probs  = np.mean(prob_smoother, axis=0)
            pred_idx   = int(np.argmax(avg_probs))
            top_action = action_names[pred_idx]
            confidence = float(avg_probs[pred_idx]) * 100

        # Build side panel
        panel = draw_panel(display.shape[0], action_names, avg_probs,
                           top_action, confidence, frame_count, len(feature_buffer))

        # Combine frame + panel
        combined = np.hstack([display, panel])

        # Status indicator top-left
        status     = "DETECTING" if pose_detected else "NO POSE"
        status_col = (0, 220, 100) if pose_detected else (60, 60, 200)
        cv2.rectangle(combined, (8, 8), (150, 28), (0, 0, 0), -1)
        cv2.putText(combined, status, (12, 23),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, status_col, 1, cv2.LINE_AA)

        cv2.imshow("Action Recognition", combined)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print(f"\nDone. Processed {frame_count} frames.")


if __name__ == "__main__":
    run()

Loading models...


/Users/sushanth/Library/jupyterlab-desktop/envs/tenso/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Running in WEBCAM mode — press Q to quit


KeyboardInterrupt: 